# 2주차. Structured Output과 Pydantic

## 0. 목표

실제 OpenAI 모델 응답을 Pydantic 모델로 받아 일정·할 일 데이터를 안정적으로 구조화한다.


## 1. 준비

`.env`의 `OPENAI_API_KEY`가 필수다.


In [ ]:
import json
import os
from typing import Any

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI

load_dotenv()

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
OPENAI_EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env 파일에 OPENAI_API_KEY를 설정한 뒤 다시 실행하세요.")


def make_model(max_tokens: int = 500) -> ChatOpenAI:
    return ChatOpenAI(model=OPENAI_MODEL, temperature=0, max_completion_tokens=max_tokens)


def show_json(value: Any) -> None:
    print(json.dumps(value, ensure_ascii=False, indent=2, default=str))


def final_text(agent_result: dict[str, Any]) -> str:
    return agent_result["messages"][-1].content


def extract_tool_trace(agent_result: dict[str, Any]) -> list[dict[str, Any]]:
    trace = []
    for message in agent_result.get("messages", []):
        for call in getattr(message, "tool_calls", []) or []:
            trace.append({"event": "tool_call", "tool_name": call.get("name"), "arguments": call.get("args", {})})
        if getattr(message, "type", None) == "tool":
            trace.append({"event": "tool_result", "tool_name": getattr(message, "name", None), "content": message.content})
    return trace

from typing import Literal
from pydantic import BaseModel, Field


## 2. 개념

Structured output은 모델의 자유로운 문장을 수업 코드가 바로 쓰기 좋은 객체로 받는 방법이다.


## 3. 기본 개념 실습

가장 작은 흐름은 "자연어 요청 → Pydantic response_format → 구조화 객체"이다. 먼저 일정, 할 일, 알 수 없는 요청만 분류한다.


In [ ]:
class ScheduleCreate(BaseModel):
    title: str = Field(description="일정 제목")
    date: str = Field(description="YYYY-MM-DD")
    start_time: str = Field(description="HH:MM")
    attendees: list[str] = Field(default_factory=list)

class TodoCreate(BaseModel):
    title: str
    due_date: str | None = Field(default=None, description="YYYY-MM-DD")
    priority: Literal["low", "medium", "high"] = "medium"

class ExtractionResult(BaseModel):
    kind: Literal["schedule", "todo", "unknown"]
    schedule: ScheduleCreate | None = None
    todo: TodoCreate | None = None
    question: str | None = None

extract_agent = create_agent(
    model=make_model(),
    tools=[],
    response_format=ExtractionResult,
    system_prompt="오늘은 2026-04-23이다. 사용자 요청을 schedule, todo, unknown 중 하나로 구조화한다.",
)

student_request = "금요일까지 보고서 제출 할 일 high"
result = extract_agent.invoke({"messages": [{"role": "user", "content": student_request}]})
structured_response = result["structured_response"]
show_json(structured_response.model_dump())


## 4. 카나메이트 확장 예제

기본 `schedule/todo` 구조에 `reminder` 타입을 추가한다. 이 확장 agent는 5번에서 실행하고, 6번 helper와 6-1 UI가 같은 structured response를 사용한다.


In [ ]:
class ReminderCreate(BaseModel):
    title: str = Field(description="알림 제목")
    related_event: str | None = Field(default=None, description="알림이 연결된 일정이나 사건")
    offset_minutes: int = Field(description="기준 사건 몇 분 전에 알릴지")

class PracticeExtractionResult(BaseModel):
    kind: Literal["schedule", "todo", "reminder", "unknown"]
    schedule: ScheduleCreate | None = None
    todo: TodoCreate | None = None
    reminder: ReminderCreate | None = None
    question: str | None = None

practice_extract_agent = create_agent(
    model=make_model(),
    tools=[],
    response_format=PracticeExtractionResult,
    system_prompt=(
        "오늘은 2026-04-23이다. 사용자 요청을 schedule, todo, reminder, unknown 중 하나로 구조화한다. "
        "'N분 전에 알려줘' 같은 요청은 reminder로 분류하고 offset_minutes에는 N을 정수로 넣는다."
    ),
)


## 5. 확장 예제 실행

추가한 `reminder` 타입으로 알림 요청을 구조화한다. 학생은 `practice_request`를 바꿔보며 Pydantic 객체의 `kind`와 세부 필드가 어떻게 달라지는지 확인한다.


In [ ]:
practice_request = "발표 30분 전에 알려줘"
practice_result = practice_extract_agent.invoke({"messages": [{"role": "user", "content": practice_request}]})
practice_response = practice_result["structured_response"]
show_json(practice_response.model_dump())

assert practice_response.kind == "reminder"
assert practice_response.reminder is not None
assert practice_response.reminder.offset_minutes == 30
print("2주차 확장 예제 실행 통과")


## 6. 코드 작성형 실습(`week02.py`)

이번 실습은 `reminder`까지 확장한 structured-output 흐름을 같은 주차 과제 파일 `week02.py`의 `run_student_structured_request`에서 구현하고 실행한다.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "week02.py").exists():
            return candidate
    raise RuntimeError("repo root를 찾지 못했습니다. 노트북을 repo 안에서 실행하세요.")


repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from week02 import extract_tool_trace, final_text, show_json, run_student_structured_request


### 실행 예시

In [ ]:
student_request = "발표 30분 전에 알려줘"
practice_response = run_student_structured_request(student_request)
show_json(practice_response.model_dump())

## 6-0. 실습 자동 점검

아래 셀은 모델 답변 문구가 아니라 trace, structured response, payload처럼 구조화된 값을 확인한다.

In [ ]:
assert practice_response.kind == "reminder"
assert practice_response.reminder is not None
assert practice_response.reminder.offset_minutes == 30
print("2주차 코드 작성형 실습 통과")

## 6-1. 로컬 Gradio UI 실습

이번 주 Gradio UI도 `week02.py` 안에 들어 있다. 터미널에서 아래 명령을 실행하면 해당 주차 UI가 바로 열린다.

```bash
python week02.py
```

노트북 안에서는 다음 셀에서 `create_demo()`로 Gradio Blocks 객체를 확인할 수 있다.

In [ ]:
from week02 import create_demo

demo = create_demo()
demo

## 7. 회고

이번 주에는 모델이 낸 결과를 바로 믿지 않고, Pydantic 객체로 검증해 다음 도구 입력으로 넘겼다.
